# 05 - Leakage-Free Model-Ready Packaging

This notebook creates the final strict-forecast model-ready files.

The goal is conservative: **zero leakage in the predictor matrix**.

Therefore this notebook exports only the safest view:

- `X`: identifiers + split + leakage-free features only.
- `y`: identifiers + split + target candidates only.

It deliberately excludes:

- same-day source uncertainty columns;
- same-day weather features;
- target columns from `X`;
- raw source rainfall columns from `X`.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import warnings

import numpy as np
import pandas as pd

try:
    display
except NameError:
    def display(obj):
        if hasattr(obj, 'to_string'):
            print(obj.to_string())
        else:
            print(obj)

warnings.filterwarnings('ignore', category=FutureWarning)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'sea_rainfall_daily_2020_2025_processed_features.csv'
MODEL_READY_DIR = PROJECT_ROOT / 'data' / 'processed' / 'model_ready'
REPORT_DIR = PROJECT_ROOT / 'reports' / '05_model_ready_packaging'
TABLE_DIR = REPORT_DIR / 'tables'

MODEL_READY_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

BASE_STEM = 'sea_rainfall_daily_2020_2025'
X_PATH = MODEL_READY_DIR / f'{BASE_STEM}_model_ready_strict_forecast_X.csv'
Y_PATH = MODEL_READY_DIR / f'{BASE_STEM}_model_ready_targets_y.csv'
FEATURE_SCHEMA_PATH = TABLE_DIR / '01_strict_forecast_feature_schema.csv'
TARGET_SCHEMA_PATH = TABLE_DIR / '02_target_schema.csv'
LEAKAGE_AUDIT_PATH = TABLE_DIR / '03_leakage_audit.csv'
REJECTED_COLUMNS_PATH = TABLE_DIR / '04_rejected_columns_not_in_X.csv'
SUMMARY_PATH = REPORT_DIR / 'MODEL_READY_PACKAGE_SUMMARY.md'
MANIFEST_PATH = REPORT_DIR / f'{BASE_STEM}_model_ready_manifest.json'

ID_COLS = ['sample_id', 'entity_id', 'date', 'split', 'country', 'location_name']

TARGET_COLS = [
    'target_nasa_power_precipitation_mm',
    'target_open_meteo_precipitation_mm',
    'target_baseline_two_source_mean_mm',
    'target_nasa_reference_consensus_mm',
    'target_open_meteo_reference_consensus_mm',
]

STRICT_FEATURE_SOURCE_COLS = [
    'canonical_latitude',
    'canonical_longitude',
    'month_sin',
    'month_cos',
    'day_of_year_sin',
    'day_of_year_cos',
    'rainfall_lag_1d_mm',
    'rainfall_lag_7d_mm',
    'rainfall_lag_30d_mm',
    'rainfall_rolling_7d_mean_prev_mm',
    'rainfall_rolling_30d_sum_prev_mm',
    'rainfall_rolling_90d_mean_prev_mm',
    'wet_spell_days_prev',
    'dry_spell_days_prev',
    'was_wet_previous_day',
    'rainfall_lag_1d_mm_was_imputed',
    'rainfall_lag_7d_mm_was_imputed',
    'rainfall_lag_30d_mm_was_imputed',
    'rainfall_rolling_7d_mean_prev_mm_was_imputed',
    'rainfall_rolling_30d_sum_prev_mm_was_imputed',
    'rainfall_rolling_90d_mean_prev_mm_was_imputed',
    'wet_spell_days_prev_was_imputed',
    'dry_spell_days_prev_was_imputed',
    'was_wet_previous_day_was_imputed',
]

QUALITY_OR_LEAKAGE_RISK_COLS = [
    'target_reference_consensus_band_mm',
    'source_bias_nasa_minus_open_meteo_mm',
    'source_abs_diff_precipitation_mm',
    'source_relative_abs_diff_precipitation',
    'source_bias_train_city_month_mm',
    'wet_day_disagreement',
    'high_source_gap_10mm',
    'high_source_gap_20mm',
]

WEATHER_SAME_DAY_COLS = [
    'temp_mean_c_mean_two_sources',
    'temp_max_c_mean_two_sources',
    'temp_min_c_mean_two_sources',
    'relative_humidity_pct_mean_two_sources',
    'wind_speed_ms_mean_two_sources',
    'surface_pressure_kpa_mean_two_sources',
]

print(f'Project root: {PROJECT_ROOT}')
print(f'Input processed file: {PROCESSED_INPUT_PATH}')
print(f'Model-ready output dir: {MODEL_READY_DIR}')

## 1. Load Processed Data

The input is the leakage-aware processed table from notebook 03. This notebook will not use every column from that table. It will export only a strict leakage-free `X` matrix and a separate `y` table.

In [ ]:
df = pd.read_csv(PROCESSED_INPUT_PATH, parse_dates=['date'])
df = df.sort_values(['entity_id', 'date']).reset_index(drop=True)
df['sample_id'] = df['entity_id'] + '__' + df['date'].dt.strftime('%Y%m%d')

required_cols = set(ID_COLS + TARGET_COLS + STRICT_FEATURE_SOURCE_COLS)
missing_required = sorted(required_cols - set(df.columns))
if missing_required:
    raise ValueError(f'Missing required columns: {missing_required}')

print('Input shape:', df.shape)
print('Rows:', len(df))
print('Entities:', df['entity_id'].nunique())
print('Date range:', df['date'].min().date(), 'to', df['date'].max().date())
display(df.head())

## 2. Build Leakage-Free X and Separate y

`X` contains only:

- identifiers and split labels;
- location coordinates;
- cyclic seasonality;
- lag/rolling rainfall history from previous days;
- previous wet/dry spell features;
- imputation flags.

`X` does not contain target columns, source uncertainty columns, or same-day weather columns.

In [ ]:
feature_rename_map = {
    column: f'feature_{column}'
    for column in STRICT_FEATURE_SOURCE_COLS
}

X = df[ID_COLS + STRICT_FEATURE_SOURCE_COLS].rename(columns=feature_rename_map).copy()
y = df[ID_COLS + TARGET_COLS].copy()

feature_cols = [feature_rename_map[column] for column in STRICT_FEATURE_SOURCE_COLS]
target_cols = TARGET_COLS[:]

print('X shape:', X.shape)
print('y shape:', y.shape)
display(X.head())
display(y.head())

## 3. Leakage Audit

The audit checks that the exported `X` matrix has no target columns, no same-day source uncertainty fields, no same-day weather fields, and that lag features truly use past rainfall only.

In [ ]:
def allclose_series(left, right):
    return bool(np.allclose(left.astype(float), right.astype(float), equal_nan=False))


audit_rows = []

def add_check(check, passed, detail):
    audit_rows.append({'check': check, 'passed': bool(passed), 'detail': detail})


add_check(
    'feature_columns_have_feature_prefix',
    all(column.startswith('feature_') for column in feature_cols),
    'Every predictor column is explicitly prefixed with feature_.',
)
add_check(
    'no_target_columns_in_X',
    not any(column.startswith('target_') for column in X.columns),
    'Target columns are stored only in y, never in X.',
)
add_check(
    'no_source_uncertainty_columns_in_X',
    not any(any(term in column for term in ['source_bias', 'source_abs_diff', 'source_relative', 'wet_day_disagreement', 'high_source_gap', 'consensus_band']) for column in X.columns),
    'Same-day source-disagreement metadata is excluded from X.',
)
add_check(
    'no_same_day_weather_columns_in_X',
    not any(any(weather_col in column for weather_col in WEATHER_SAME_DAY_COLS) for column in X.columns),
    'Same-day weather summary columns are excluded from strict-forecast X.',
)

grouped_target = df.groupby('entity_id')['target_baseline_two_source_mean_mm']

lag1_expected = grouped_target.shift(1)
mask = df['rainfall_lag_1d_mm_was_imputed'].eq(0)
add_check(
    'rainfall_lag_1d_uses_previous_day_only',
    allclose_series(df.loc[mask, 'rainfall_lag_1d_mm'], lag1_expected.loc[mask]),
    'Non-imputed lag-1 rainfall equals previous-day baseline rainfall within each entity.',
)

lag7_expected = grouped_target.shift(7)
mask = df['rainfall_lag_7d_mm_was_imputed'].eq(0)
add_check(
    'rainfall_lag_7d_uses_previous_7th_day_only',
    allclose_series(df.loc[mask, 'rainfall_lag_7d_mm'], lag7_expected.loc[mask]),
    'Non-imputed lag-7 rainfall equals the value seven days earlier within each entity.',
)

lag30_expected = grouped_target.shift(30)
mask = df['rainfall_lag_30d_mm_was_imputed'].eq(0)
add_check(
    'rainfall_lag_30d_uses_previous_30th_day_only',
    allclose_series(df.loc[mask, 'rainfall_lag_30d_mm'], lag30_expected.loc[mask]),
    'Non-imputed lag-30 rainfall equals the value thirty days earlier within each entity.',
)

prior_rainfall = grouped_target.shift(1)
rolling_7_expected = prior_rainfall.groupby(df['entity_id']).rolling(7, min_periods=3).mean().reset_index(level=0, drop=True)
mask = df['rainfall_rolling_7d_mean_prev_mm_was_imputed'].eq(0)
add_check(
    'rolling_7d_mean_uses_prior_days_only',
    allclose_series(df.loc[mask, 'rainfall_rolling_7d_mean_prev_mm'], rolling_7_expected.loc[mask]),
    'Rolling 7-day mean is computed from prior rainfall values only.',
)

rolling_30_expected = prior_rainfall.groupby(df['entity_id']).rolling(30, min_periods=10).sum().reset_index(level=0, drop=True)
mask = df['rainfall_rolling_30d_sum_prev_mm_was_imputed'].eq(0)
add_check(
    'rolling_30d_sum_uses_prior_days_only',
    allclose_series(df.loc[mask, 'rainfall_rolling_30d_sum_prev_mm'], rolling_30_expected.loc[mask]),
    'Rolling 30-day sum is computed from prior rainfall values only.',
)

rolling_90_expected = prior_rainfall.groupby(df['entity_id']).rolling(90, min_periods=30).mean().reset_index(level=0, drop=True)
mask = df['rainfall_rolling_90d_mean_prev_mm_was_imputed'].eq(0)
add_check(
    'rolling_90d_mean_uses_prior_days_only',
    allclose_series(df.loc[mask, 'rainfall_rolling_90d_mean_prev_mm'], rolling_90_expected.loc[mask]),
    'Rolling 90-day mean is computed from prior rainfall values only.',
)

add_check(
    'X_has_no_missing_values',
    int(X.isna().sum().sum()) == 0,
    f'X missing cells: {int(X.isna().sum().sum())}.',
)
add_check(
    'y_has_no_missing_values',
    int(y.isna().sum().sum()) == 0,
    f'y missing cells: {int(y.isna().sum().sum())}.',
)
add_check(
    'sample_id_unique',
    X['sample_id'].is_unique and y['sample_id'].is_unique,
    'sample_id is unique in X and y.',
)
add_check(
    'X_y_alignment',
    X['sample_id'].equals(y['sample_id']),
    'X and y rows are aligned by sample_id.',
)

leakage_audit = pd.DataFrame(audit_rows)
if not leakage_audit['passed'].all():
    display(leakage_audit)
    raise ValueError('Leakage audit failed. Do not export model-ready data.')

display(leakage_audit)

## 4. Document Included and Rejected Columns

The rejected list explains why the strict model-ready `X` does not include several processed columns.

In [ ]:
feature_schema = pd.DataFrame([
    {
        'feature_column': feature_rename_map[source_column],
        'source_column': source_column,
        'feature_group': (
            'location' if source_column in ['canonical_latitude', 'canonical_longitude']
            else 'calendar'
            if source_column in ['month_sin', 'month_cos', 'day_of_year_sin', 'day_of_year_cos']
            else 'imputation_flag'
            if source_column.endswith('_was_imputed')
            else 'rainfall_history'
        ),
        'leakage_status': 'safe_for_strict_forecast_X',
    }
    for source_column in STRICT_FEATURE_SOURCE_COLS
])

target_schema = pd.DataFrame([
    {
        'target_column': column,
        'target_group': 'rainfall_target_candidate',
        'note': 'Stored in y only; never included in X.',
    }
    for column in TARGET_COLS
])

rejected_rows = []
for column in df.columns:
    if column in ID_COLS or column in TARGET_COLS or column in STRICT_FEATURE_SOURCE_COLS:
        continue
    if column in QUALITY_OR_LEAKAGE_RISK_COLS:
        reason = 'same_day_source_uncertainty_or_quality_metadata_not_allowed_in_X'
    elif column in WEATHER_SAME_DAY_COLS or column.endswith('_source_abs_diff'):
        reason = 'same_day_weather_or_weather_source_gap_excluded_from_strict_forecast_X'
    else:
        reason = 'metadata_or_intermediate_column_not_needed_for_strict_forecast_X'
    rejected_rows.append({'column_name': column, 'reason': reason})

rejected_columns = pd.DataFrame(rejected_rows)

display(feature_schema)
display(target_schema)
display(rejected_columns.head(30))

## 5. Save Model-Ready Package

The output files are placed under `data/processed/model_ready/` and named explicitly for training.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


X.to_csv(X_PATH, index=False)
y.to_csv(Y_PATH, index=False)
feature_schema.to_csv(FEATURE_SCHEMA_PATH, index=False)
target_schema.to_csv(TARGET_SCHEMA_PATH, index=False)
leakage_audit.to_csv(LEAKAGE_AUDIT_PATH, index=False)
rejected_columns.to_csv(REJECTED_COLUMNS_PATH, index=False)

summary_lines = [
    '# Leakage-Free Model-Ready Package',
    '',
    '## Output Files',
    f'- X matrix: `data/processed/model_ready/{X_PATH.name}`',
    f'- y targets: `data/processed/model_ready/{Y_PATH.name}`',
    '',
    '## Leakage Policy',
    '- X contains only strict forecast features: location, cyclic calendar, previous rainfall history, previous wet/dry spell features, and imputation flags.',
    '- X contains no target columns.',
    '- X contains no same-day source uncertainty columns.',
    '- X contains no same-day weather columns.',
    '- y contains target candidates only and is stored separately.',
    '',
    '## Row Alignment',
    '- X and y share the same `sample_id`, `entity_id`, `date`, and `split` columns.',
    '- Use `sample_id` or `(entity_id, date)` to join predictions back to targets.',
]
SUMMARY_PATH.write_text('\n'.join(summary_lines), encoding='utf-8')

manifest = {
    'dataset_name': 'Leakage-free strict forecast model-ready rainfall package',
    'created_by_notebook': 'notebooks/05_model_ready_packaging.ipynb',
    'created_at_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'input_file': str(PROCESSED_INPUT_PATH.relative_to(PROJECT_ROOT)),
    'output_files': {
        'X': str(X_PATH.relative_to(PROJECT_ROOT)),
        'y': str(Y_PATH.relative_to(PROJECT_ROOT)),
    },
    'row_counts': {
        'X_rows': int(len(X)),
        'y_rows': int(len(y)),
        'train_rows': int((X['split'] == 'train').sum()),
        'validation_rows': int((X['split'] == 'validation').sum()),
        'test_rows': int((X['split'] == 'test').sum()),
    },
    'column_counts': {
        'X_columns_total': int(X.shape[1]),
        'X_feature_columns': int(len(feature_cols)),
        'y_columns_total': int(y.shape[1]),
        'target_columns': int(len(TARGET_COLS)),
    },
    'leakage_audit': leakage_audit.to_dict(orient='records'),
    'files': [
        {'path': str(path.relative_to(PROJECT_ROOT)), 'sha256': sha256_file(path)}
        for path in [X_PATH, Y_PATH, FEATURE_SCHEMA_PATH, TARGET_SCHEMA_PATH, LEAKAGE_AUDIT_PATH, REJECTED_COLUMNS_PATH, SUMMARY_PATH]
    ],
}

MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding='utf-8')

print('Saved leakage-free model-ready files:')
print('-', X_PATH.relative_to(PROJECT_ROOT), '| shape=', X.shape)
print('-', Y_PATH.relative_to(PROJECT_ROOT), '| shape=', y.shape)
print('\nSaved report:')
for path in [FEATURE_SCHEMA_PATH, TARGET_SCHEMA_PATH, LEAKAGE_AUDIT_PATH, REJECTED_COLUMNS_PATH, SUMMARY_PATH, MANIFEST_PATH]:
    print('-', path.relative_to(PROJECT_ROOT))